# Đánh giá Agentic RAG Refined từ dữ liệu Naive RAG (150 câu)

Notebook này tái xử lý kết quả từ file `evaluation_04_naive_rag.csv` theo chiến lược 2 tầng:

- Nếu câu trả lời cũ đã ổn (`score > 1`) thì giữ nguyên.
- Nếu `score == 1` thì chạy Agentic RAG.

Chỉ các câu có `process_level` là 1 hoặc 2 mới được chấm lại bằng Judge LLM.

## 1. Import thư viện và khởi tạo backend

In [11]:
import os
import sys
import time
import json
import re
import random
import pandas as pd

from dotenv import load_dotenv
from groq import RateLimitError
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

sys.path.append(os.path.abspath('..'))
from backend.llm_agent import run_agentic_rag, init_agents
from backend.vector_store import get_retriever

INPUT_CSV = os.path.join('..', 'data', 'evaluation', 'evaluation_04_naive_rag.csv')
OUTPUT_CSV = os.path.join('..', 'data', 'evaluation', 'evaluation_05_agentic_rag_refined.csv')
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)

init_agents()
get_retriever()
load_dotenv('../.env', override=True)

GROQ_KEYS = []
default_key = os.getenv('GROQ_API_KEY')
if default_key: GROQ_KEYS.append(default_key)
i = 1
while True:
    k = os.getenv(f'GROQ_API_KEY_{i}')
    if not k: break
    if k not in GROQ_KEYS: GROQ_KEYS.append(k)
    i += 1

random.shuffle(GROQ_KEYS)
current_judge_idx = 0
judge_llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0, api_key=GROQ_KEYS[0])
print(f'Hệ thống sẵn sàng với {len(GROQ_KEYS)} keys (đã shuffle).')


🔑 [Agent] Tìm thấy 10 Groq API Key để xoay vòng.
⏳ [Agent] Đang kết nối mạng lưới đa Mô hình Groq...
   🔧 [Rebuild] Tất cả chain đã được tạo lại với Key #1: ...p9JBAl
✅ [Agent] Hệ thống Multi Agent đã sẵn sàng!
Hệ thống sẵn sàng với 10 keys (đã shuffle).


## 2. Đọc dữ liệu từ file đánh giá Naive RAG (04)

In [12]:
df_source = pd.read_csv(INPUT_CSV, encoding='utf-8-sig')
if 'question_id' not in df_source.columns and 'id' in df_source.columns: 
    df_source = df_source.rename(columns={'id': 'question_id'})
df_source['score'] = pd.to_numeric(df_source['score'], errors='coerce').fillna(0)
if 'reasoning' not in df_source.columns: df_source['reasoning'] = ''
print(f'Đã nạp {len(df_source)} dòng từ file 04.')

Đã nạp 150 dòng từ file 04.


## 3. Hàm hỗ trợ: Judge LLM và Utilities

In [ ]:
def rotate_judge(reason: str = ''):
    global current_judge_idx, judge_llm
    current_judge_idx = (current_judge_idx + 1) % len(GROQ_KEYS)
    judge_llm = ChatGroq(model='openai/gpt-oss-120b', temperature=0, api_key=GROQ_KEYS[current_judge_idx])
    print(f'Đổi Judge key #{current_judge_idx + 1} ({reason})')

def evaluate_answer(question: str, ground_truth: str, generated_answer: str) -> dict:
    global judge_llm
    tried = 0
    while tried < len(GROQ_KEYS):
        try:
            prompt = ChatPromptTemplate.from_template(
                """Bạn là một giám khảo công bằng, chuyên đánh giá chất lượng câu trả lời của AI.
Nhiệm vụ: So sánh câu trả lời của AI với câu trả lời chuẩn (Ground Truth) cho câu hỏi dưới đây.
Tiêu chí chấm điểm (thang điểm 1-10):
- Điểm 8-10: Câu trả lời AI truyền đạt đúng ý nghĩa và thông tin chính như Ground Truth, có thể diễn đạt khác nhưng nội dung chính xác.
- Điểm 4-7: Câu trả lời AI thiếu một số chi tiết quan trọng, hoặc có thông tin không hoàn toàn chính xác.
- Điểm 1-3: Câu trả lời AI sai hoàn toàn, không liên quan, hoặc bịa đặt thông tin.
Câu hỏi:
{question}
Ground Truth (Câu trả lời chuẩn):
{ground_truth}
Câu trả lời của AI:
{generated_answer}
Hãy đánh giá và trả về kết quả CHÍNH XÁC theo định dạng JSON sau (không thêm bất kỳ văn bản nào khác):
{{"score": <số nguyên từ 1-10>, "reasoning": "<lý do ngắn gọn bằng tiếng Việt>"}}
"""
            )

            resp = judge_llm.invoke(prompt.format(
                question=question, ground_truth=ground_truth, generated_answer=generated_answer))
            raw = resp.content if hasattr(resp, 'content') else str(resp)
            
            # Tách JSON bằng Regex (giúp tránh các lời dẫn thừa của AI)
            match = re.search(r'\{.*\}', raw.replace('\n', ' '), flags=re.DOTALL)
            if not match:
                print(f"[ERROR] Không tìm thấy JSON trong output: {raw}")
                return {'score': 0, 'reasoning': 'Lỗi format đầu ra của AI'}
            
             # Parse JSON và ép kiểu điểm số
            data = json.loads(match.group(0))
            return {
                'score': int(data.get('score', 0)),
                'reasoning': str(data.get('reasoning', '')).strip()
            }
            
        except Exception as e:
            print(f"Lỗi Judge tại QID: {e}")
            # Nếu lỗi "Expecting ',' delimiter", in ra raw để debug
            if "Expecting" in str(e):
                print(f"   AI trả về lỗi: {raw}")
            
            rotate_judge('Rate limit/Error')
            tried += 1
            
    return {'score': 0, 'reasoning': 'Fail to judge'}

def check_done(row) -> bool:
    """
    Kiểm tra xem một dòng đã hoàn thành đầy đủ kết quả chưa.
    Một câu được coi là XONG nếu:
    - answer: Không trống
    - score: Có giá trị và lớn hơn 0
    - reasoning: Không trống
    """
    # 1. Kiểm tra câu trả lời
    answer = str(row.get('generated_answer', '')).strip()
    has_answer = answer not in ('', 'nan', 'None')
    
    # 2. Kiểm tra điểm số (Phải > 0 mới là đã chấm thành công)
    score_val = row.get('score')
    has_score = pd.notna(score_val) and float(score_val or 0) > 0
    
    # 3. Kiểm tra lý do chấm điểm
    reason = str(row.get('reasoning', '')).strip()
    has_reason = reason not in ('', 'nan', 'None')
    
    return has_answer and has_score and has_reason

## 4. Xử lý 2 tầng Agentic RAG và gán `process_level`

In [48]:
# Chỉ nạp từ output nếu đã tồn tại, tránh đụng vào input cũ
if os.path.exists(OUTPUT_CSV):
    df_result = pd.read_csv(OUTPUT_CSV, encoding='utf-8-sig')
    # Chuyển score sang số, giữ nguyên NaN nếu là ô trống
    df_result['score'] = pd.to_numeric(df_result['score'], errors='coerce')
    print(f'Đang tiếp tục làm việc trên file: {OUTPUT_CSV} ({len(df_result)} dòng)')
else:
    # Nếu chưa có file output thì mới nạp từ source 04
    df_source = pd.read_csv(INPUT_CSV, encoding='utf-8-sig')
    df_result = df_source.copy()
    if 'id' in df_result.columns: df_result = df_result.rename(columns={'id': 'question_id'})
    df_result['process_level'] = 0
    df_result['latency_seconds'] = 0.0
    print('Khởi tạo file kết quả mới.')

for idx, row in df_result.iterrows():
    # 1. Lấy thông tin hiện tại
    qid = row['question_id']
    question = str(row['question'])
    current_answer = str(row.get('generated_answer', '')).strip()
    
    # Kiểm tra xem ô này có bị bạn xóa trống hay không
    has_already_generated = current_answer not in ('', 'nan', 'None')
    
    src_rows = df_source[df_source['question_id'] == qid]
    naive_score = float(src_rows['score'].values[0]) if len(src_rows) > 0 else 0.0
    process_level = int(row.get('process_level', 0))

    # 2. XÁC ĐỊNH CHÍNH XÁC KHI NÀO CHẠY AGENTIC
    # Chỉ chạy Agentic khi: 
    #   - Answer bị xóa trống (bạn ép nó đi tìm câu trả lời mới)
    #   - Hoặc đây là câu lấy nguyên gốc từ Naive RAG (process_level = 0) và có điểm cực thấp <= 1.
    should_run_agentic = (not has_already_generated) or (naive_score <= 1.0 and process_level == 0)

    # Nếu KHÔNG chạy Agentic thì BỎ QUA lun cell này (để dành đó cho Cell 5 chấm điểm)
    if not should_run_agentic:
        continue

    # 3. CHỈ KHI CẦN TẠO MỚI ANSWER MỚI CHẠY ĐOẠN NÀY DƯỚI
    print(f'\n--- [{idx + 1}/{len(df_result)}] Xử lý Agentic QID={qid} ---')
    t_start = time.time()
    ans, _, _, retries, is_found, _ = run_agentic_rag(question, max_retries=3, skip_router=True)
    latency = round(time.time() - t_start, 2)
    
    # Cập nhật kết quả sinh văn bản mới vào Dataframe
    df_result.at[idx, 'generated_answer'] = ans
    df_result.at[idx, 'process_level'] = 1 if retries <= 1 else 2
    df_result.at[idx, 'latency_seconds'] = latency
    
    # CHỈ RESET SCORE VỀ 0 NẾU NÓ TẠO RA CÂU TRẢ LỜI MỚI (Để báo judge chấm)
    df_result.at[idx, 'score'] = 0 
    df_result.at[idx, 'reasoning'] = ''
    print(f'   Câu trả lời của AI: {ans}')

    status = 'thấy' if is_found else 'không thấy'
    print(f'   -> Kết quả mới: {status} (retries={retries}, level={df_result.at[idx, "process_level"]})')
    
    save_checkpoint(df_result)

print('\n✅ Hoàn tất quá trình sinh văn bản. Những câu còn trống Score hãy chạy Cell Judge để chấm.')

Đang tiếp tục làm việc trên file: ..\data\evaluation\evaluation_05_agentic_rag_refined.csv (150 dòng)

--- [5/150] Xử lý Agentic QID=5 ---
🔄 [Rewriter Loop 0/3] Đang quét với Query: Chứng chỉ Tiếng Nhật JLAN J4 đạt 91 điểm được quy đổi điểm học phần Tiếng Nhật 1 là bao nhiêu?
⚠️ Context không đủ dữ kiện, Model đang lên kế hoạch viết lại từ khóa search...
✨ Rewritten Query đổi thành: 'Chuyển đổi 91 điểm chứng chỉ JLAN J4 sang điểm môn Tiếng Nhật 1 như thế nào?'
🔄 [Rewriter Loop 1/3] Đang quét với Query: Chuyển đổi 91 điểm chứng chỉ JLAN J4 sang điểm môn Tiếng Nhật 1 như thế nào?
⚠️ Context không đủ dữ kiện, Model đang lên kế hoạch viết lại từ khóa search...
✨ Rewritten Query đổi thành: 'Tỷ lệ chuyển đổi 91 điểm JLAN J4 sang điểm học phần Tiếng Nhật 1'
🔄 [Rewriter Loop 2/3] Đang quét với Query: Tỷ lệ chuyển đổi 91 điểm JLAN J4 sang điểm học phần Tiếng Nhật 1
⚠️ Context không đủ dữ kiện, Model đang lên kế hoạch viết lại từ khóa search...
✨ Rewritten Query đổi thành: 'JLAN J4 score convers

## 5. Chấm lại điểm cho các câu cần thiết

In [49]:
# Lọc ra những câu chưa đạt điều kiện 'check_done'
mask_rejudge = df_result.apply(lambda r: not check_done(r), axis=1)
df_rejudge_list = df_result[mask_rejudge]

print(f'Số câu cần chấm lại (bao gồm cả câu lỗi hoặc chưa chấm): {len(df_rejudge_list)}')

for idx_row in df_rejudge_list.index:
    # Lấy object "dòng" hiện tại một cách tường minh
    current_row = df_result.loc[idx_row]

    # Lấy answer và QID trực tiếp từ object này, không dùng tên biến trùng với Cell 4
    current_ans = str(current_row.get('generated_answer', '')).strip()
    current_qid = current_row.get("question_id", "Không rõ QID") # Lấy đúng QID của dòng này

    print(f'\n--- Đang xử lý QID={current_qid} (Dòng Index={idx_row}) ---')

    # 1. Kiểm tra đầu vào của Answer
    if not ans or ans in ('nan', 'None'):
        print(f'   ⚠️ QID={current_qid} lỗi: Answer đang bị TRỐNG. Hãy chạy Cell 4 trước.')
        continue

    # 2. Gọi Judge và in kết quả thô
    print(f'   🔍 Đang gọi Judge chấm điểm...')
    ev = evaluate_answer(str(current_row['question']), str(current_row['ground_truth']), ans)
    
    print(f'   📝 Kết quả Judge trả về: Score={ev.get("score")}, Reason={ev.get("reasoning")}')
    # 3. Cập nhật và kiểm tra xem check_done đã hài lòng chưa
    df_result.at[idx_row, 'score'] = ev.get('score', 0)
    df_result.at[idx_row, 'reasoning'] = ev.get('reasoning', '')
    
    if check_done(df_result.loc[idx_row]):
        print(f'   ✅ Thành công: Câu QID={current_qid} đã đạt chuẩn Done.')
    else:
        print(f'   ❌ Thất bại: Câu QID={current_qid} vẫn CHƯA đạt chuẩn check_done (do Score=0 hoặc Reason trống).')
    
    # 4. Lưu ngay lập tức
    save_checkpoint(df_result)
print('✅ Chấm điểm hoàn tất.')

Số câu cần chấm lại (bao gồm cả câu lỗi hoặc chưa chấm): 1

--- Đang xử lý QID=5 (Dòng Index=4) ---
   🔍 Đang gọi Judge chấm điểm...
   📝 Kết quả Judge trả về: Score=1, Reason=AI không đưa ra đáp án 10 điểm như Ground Truth mà chỉ đưa thông tin không liên quan và không trả lời câu hỏi.
   ✅ Thành công: Câu QID=5 đã đạt chuẩn Done.
✅ Chấm điểm hoàn tất.


## 6. Tổng kết nhanh kết quả

In [55]:
# --- CELL 6: THỐNG KÊ CHI TIẾT THEO TẦNG XỬ LÝ ---
print('=' * 80)
print('📊 KẾT QUẢ ĐÁNH GIÁ AGENTIC RAG REFINED')
print('=' * 80)

# Tổng số câu
total_rows = len(df_result)
print(f"Tổng số câu hỏi đánh giá: {total_rows}")

# Thống kê theo từng Level
if 'process_level' in df_result.columns and 'score' in df_result.columns:
    stats = df_result.groupby('process_level')['score'].agg(['count', 'mean']).reset_index()
    
    # Mapping tên level cho dễ đọc
    level_names = {
        0: "Level 0 (Naive - Dùng kết quả cũ)",
        1: "Level 1 (Agentic - Quét Default)",
        2: "Level 2 (Agentic - Quét Thinking)"
    }
    
    for _, row_stat in stats.iterrows():
        lv = int(row_stat['process_level'])
        name = level_names.get(lv, f"Level {lv}")
        count = int(row_stat['count'])
        avg = row_stat['mean']
        percentage = (count / total_rows) * 100
        print(f"\n🔹 {name}:")
        print(f"   - Số lượng: {count} câu ({percentage:.1f}%)")
        print(f"   - Điểm TB : {avg:.2f} / 10")

print('\n' + '-' * 80)
overall_avg = df_result['score'].mean()
print(f"🏆 ĐIỂM TRUNG BÌNH TOÀN BỘ HỆ THỐNG: {overall_avg:.2f} / 10")
print('=' * 80)

# Hiển thị 10 câu có điểm thấp nhất để bạn dễ dàng soi lỗi
print("\n📝 Top 10 câu điểm thấp nhất (để debug):")
low_scores = df_result.sort_values(by='score').head(10)
display(low_scores[['question_id', 'question', 'score', 'process_level']])


📊 KẾT QUẢ ĐÁNH GIÁ AGENTIC RAG REFINED
Tổng số câu hỏi đánh giá: 150

🔹 Level 0 (Naive - Dùng kết quả cũ):
   - Số lượng: 100 câu (66.7%)
   - Điểm TB : 8.96 / 10

🔹 Level 1 (Agentic - Quét Default):
   - Số lượng: 36 câu (24.0%)
   - Điểm TB : 4.47 / 10

🔹 Level 2 (Agentic - Quét Thinking):
   - Số lượng: 14 câu (9.3%)
   - Điểm TB : 3.29 / 10

--------------------------------------------------------------------------------
🏆 ĐIỂM TRUNG BÌNH TOÀN BỘ HỆ THỐNG: 7.35 / 10

📝 Top 10 câu điểm thấp nhất (để debug):


,question_id,question,score,process_level
4,5,Chứng chỉ Tiếng Nhật JLAN J4 đạt 91 điểm được ...,1.0,2
30,31,Sinh viên hệ đại trà K2023 làm đơn xin kéo dài...,1.0,2
45,46,Em mới trúng tuyển Khóa 2025 ngành Công nghệ t...,1.0,2
61,62,Địa chỉ website để đăng ký môn học là gì?,1.0,1
48,49,Cuối học kỳ em lười nên bỏ qua việc làm khảo s...,1.0,2
49,50,Sinh viên hệ Đại học liên thông từ hệ Cao đẳng...,1.0,2
39,40,Sinh viên tham gia biểu diễn văn nghệ trong hộ...,1.0,1
118,119,điểm qt 2 cột dc ko,1.0,1
111,112,tttn ngành cntt mấy tc,1.0,1
147,148,bị khiển trách có dc loại xuất sắc ko,1.0,1
